# TradeSense · Component 03 · notebook 51: are the explanations faithful? SHAP (XGBoost) and gradient x input (Transformer) (protocol E2 v1b; CPU, minutes; no training)

**Question.** When the dashboard says "this alert is driven mainly by *trade_flow_imbalance*", is that true for the model? We test it by *deleting* each feature (set it to its normal level) and checking that the feature the explanation names is the one whose removal lowers the model score most. Done for XGBoost (TreeSHAP) and for the Transformer (gradient × input, **built here**; it did not exist before). The Autoencoder explanation is exact (its score is the largest per-feature error). TGNN: interface only, nothing is produced or claimed.

Development data only (CAL-3 injected events); TEST-3 / TEST-B are not read. No model, feature, normalisation or threshold is changed.

**Protocol text hashed by this notebook:**


```
# Protocol E2 v1b - are the explanations faithful? SHAP for XGBoost and gradient x input for the Transformer (novelty 3)
Written BEFORE any result of this study exists. Amendment 1 (v1 -> v1b, made before any hypothesis result existed): the first run of v1 stopped at the exactness check T03, because the saved Transformer scores (made on a different device) differ from the CPU recomputation by 1.7e-3, above the 1e-3 declared for v1. The Transformer tolerance in H-C11 is therefore set to 5e-3 (the sibling notebook 32 used 2e-2 for the same parity); the XGBoost tolerance stays 1e-3. Nothing else changed. Nothing frozen is changed (models, 6 features, normalisation, fusion stacker, thresholds q). Development data only: CAL-3 injected events (replica 0, August); TEST-3 / TEST-B are not read. An explanation is faithful if the feature it names as most important really is the one whose removal hurts the model's score most.

## What is explained and how
- XGBoost v2 (declared boosters, seeds averaged): TreeSHAP (pred_contribs) of the 53 rolling columns, summed to the six base features (a cross-feature group "x" is reported but is not a deletable feature).
- Transformer v2 (declared nets, seeds averaged): gradient x input of the model output at the last position with respect to the normalised input window (W x 6); feature attribution = sum over the window of |gradient x input|; time attribution = the same per bar.
- AE-ctx: exact (the score is the largest of the six per-feature scaled errors), no approximation, not tested here.
- TGNN: interface only. No TGNN explanation is produced or claimed; the interface returns available = false with a reason until the TGNN is integrated.

## Events and the deletion test (declared now)
Events = the 400 CAL-3 replica-0 events (200 per symbol). The explained bar of an event is the bar with the highest saved score of that model inside the event. An event is MODEL-ACTIVE for a model if that score is above the model's own rolling alert line (budget 2, declared q, genuine rolling threshold at the same bar). Only model-active events enter the faithfulness hypotheses of that model; the number of excluded events is reported.
Deletion: for a feature j, set its N-normalised value to 0 (its normal level) over the last 64 bars up to and including the explained bar, recompute the model output (XGBoost: rebuild the 53 features and take the mean booster margin; Transformer: mean output of the nets). D_j = output before minus output after (positive = the model score drops). Random-feature baseline: the mean D over the six features.

## Hypotheses
- H-C5 (XGBoost SHAP is faithful): over model-active events, (a) the share of events in which the top-1 SHAP feature has the largest D among the six features is >= 0.50 (chance 1/6), AND (b) the mean over events of the Spearman correlation between the six SHAP values and the six D_j is >= 0.40 with an event-bootstrap 95% lower bound > 0.
- H-C9 (Transformer gradient x input is faithful): the same two conditions (a) and (b) for the Transformer attribution.
- H-C10 (descriptive, no threshold): share of events whose top-1 feature lies in the family's expected signature (declared in notebook 45), for the Transformer, versus chance.
- H-C11 (exactness): SHAP additivity (contributions + bias = booster margin) <= 1e-3; the recomputed XGBoost output equals the saved score on the explained bars (<= 1e-3) and the recomputed Transformer output equals the saved score (<= 5e-3, device difference); the first-order check of the gradient (gradient . delta versus the output change for a tiny delta) agrees within 5%.
- Descriptive: share of Transformer attribution in the last 10 bars of the window; faithfulness per family; the same measures for a random-attribution baseline (a random feature order).

## Known weaknesses, stated in advance
Zero-replacement leaves the other features and the learned interactions in place, so deletion measures marginal rather than unique importance; gradient x input is a local first-order attribution and may disagree with a large deletion. Both may fail and will be reported as failed. Labels and event signatures are synthetic; the explanations describe the model's score, not the cause of a real market event.

```

In [ ]:
import os, sys, json, glob, time, hashlib, warnings, datetime
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
import torch, xgboost as xgb
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 300 if MOCK else 3000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
PROTO_E2 = r"""# Protocol E2 v1b - are the explanations faithful? SHAP for XGBoost and gradient x input for the Transformer (novelty 3)
Written BEFORE any result of this study exists. Amendment 1 (v1 -> v1b, made before any hypothesis result existed): the first run of v1 stopped at the exactness check T03, because the saved Transformer scores (made on a different device) differ from the CPU recomputation by 1.7e-3, above the 1e-3 declared for v1. The Transformer tolerance in H-C11 is therefore set to 5e-3 (the sibling notebook 32 used 2e-2 for the same parity); the XGBoost tolerance stays 1e-3. Nothing else changed. Nothing frozen is changed (models, 6 features, normalisation, fusion stacker, thresholds q). Development data only: CAL-3 injected events (replica 0, August); TEST-3 / TEST-B are not read. An explanation is faithful if the feature it names as most important really is the one whose removal hurts the model's score most.

## What is explained and how
- XGBoost v2 (declared boosters, seeds averaged): TreeSHAP (pred_contribs) of the 53 rolling columns, summed to the six base features (a cross-feature group "x" is reported but is not a deletable feature).
- Transformer v2 (declared nets, seeds averaged): gradient x input of the model output at the last position with respect to the normalised input window (W x 6); feature attribution = sum over the window of |gradient x input|; time attribution = the same per bar.
- AE-ctx: exact (the score is the largest of the six per-feature scaled errors), no approximation, not tested here.
- TGNN: interface only. No TGNN explanation is produced or claimed; the interface returns available = false with a reason until the TGNN is integrated.

## Events and the deletion test (declared now)
Events = the 400 CAL-3 replica-0 events (200 per symbol). The explained bar of an event is the bar with the highest saved score of that model inside the event. An event is MODEL-ACTIVE for a model if that score is above the model's own rolling alert line (budget 2, declared q, genuine rolling threshold at the same bar). Only model-active events enter the faithfulness hypotheses of that model; the number of excluded events is reported.
Deletion: for a feature j, set its N-normalised value to 0 (its normal level) over the last 64 bars up to and including the explained bar, recompute the model output (XGBoost: rebuild the 53 features and take the mean booster margin; Transformer: mean output of the nets). D_j = output before minus output after (positive = the model score drops). Random-feature baseline: the mean D over the six features.

## Hypotheses
- H-C5 (XGBoost SHAP is faithful): over model-active events, (a) the share of events in which the top-1 SHAP feature has the largest D among the six features is >= 0.50 (chance 1/6), AND (b) the mean over events of the Spearman correlation between the six SHAP values and the six D_j is >= 0.40 with an event-bootstrap 95% lower bound > 0.
- H-C9 (Transformer gradient x input is faithful): the same two conditions (a) and (b) for the Transformer attribution.
- H-C10 (descriptive, no threshold): share of events whose top-1 feature lies in the family's expected signature (declared in notebook 45), for the Transformer, versus chance.
- H-C11 (exactness): SHAP additivity (contributions + bias = booster margin) <= 1e-3; the recomputed XGBoost output equals the saved score on the explained bars (<= 1e-3) and the recomputed Transformer output equals the saved score (<= 5e-3, device difference); the first-order check of the gradient (gradient . delta versus the output change for a tiny delta) agrees within 5%.
- Descriptive: share of Transformer attribution in the last 10 bars of the window; faithfulness per family; the same measures for a random-attribution baseline (a random feature order).

## Known weaknesses, stated in advance
Zero-replacement leaves the other features and the learned interactions in place, so deletion measures marginal rather than unique importance; gradient x input is a local first-order attribution and may disagree with a large deletion. Both may fail and will be reported as failed. Labels and event signatures are synthetic; the explanations describe the model's score, not the cause of a real market event.
"""
PHE = hashlib.sha256(PROTO_E2.encode()).hexdigest(); pp = OUT/'protocol_e2_v1b.md'
if pp.exists(): assert pp.read_text() == PROTO_E2, 'protocol E2 exists and differs from this notebook'
else: pp.write_text(PROTO_E2)
(OUT/'protocol_e2_v1b.sha256').write_text(PHE + '  protocol_e2_v1b.md\n')
for f in ['declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_explain_v1.json', 'b2_lib.py', 'c03_infer.py']: assert (OUT/f).exists(), f'{f} missing'
sys.path.insert(0, str(OUT)); import b2_lib as B; import c03_infer as c
torch.set_num_threads(2)
say('protocol E2 v1b sha256 ' + PHE[:16] + '; protocol B2 intact ' + ph[:16] + '; library ' + sha_file(OUT/'c03_infer.py') + '; xgboost ' + xgb.__version__ + ', torch ' + torch.__version__ + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol E2 v1b sha256 52c6fc77a8d2fe67; protocol B2 intact 443f99d5faf5929f; library 3b5d9df23521adf9; xgboost 3.4.1, torch 2.11.0+cpu


## 1 · Frozen models, CAL-3 events, saved scores

In [ ]:
AOLEN = 31 * 1440; WARM = B.WARM; DELLEN = 64; PRE = 130
DF = json.load(open(OUT/'declared_final_set_v1.json')); MODELS = c.MODEL_NAMES; MODES = {m: DF['models'][m]['normalisation'] for m in MODELS}; FILES = {MODELS[0]: 'ae_ctx_agg', MODELS[1]: 'xgb_v2', MODELS[2]: 'transformer_v2'}
SIGN = {1: ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'volume_acceleration', 'large_trade_count'], 2: ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'volume_acceleration', 'large_trade_count'],
        3: ['volume_acceleration', 'large_trade_count', 'rush_order_density'], 4: ['trade_flow_imbalance', 'log_returns'], 5: ['rush_order_density', 'hl_spread_ratio', 'trade_flow_imbalance'],
        6: ['hl_spread_ratio', 'log_returns', 'large_trade_count'], 7: ['log_returns', 'volume_acceleration', 'large_trade_count', 'trade_flow_imbalance', 'hl_spread_ratio'], 8: ['trade_flow_imbalance', 'log_returns', 'large_trade_count', 'volume_acceleration']}
FEATS = B.FEATS; BASEOF = [n.split('__')[0] for n in c.FA_NAMES]; GROUPS = FEATS + ['x']; GIDX = {g_: [i for i, n in enumerate(BASEOF) if n == g_] for g_ in GROUPS}; assert sum(len(v) for v in GIDX.values()) == 53
def find1(pat):
    cc = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert cc, pat; return sorted(cc)[0]
N0 = {}; AEC = {}; BUN = {}; FR = {}; EV = {}; SAV = {}; THR = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    N0[s] = B.fit_n0(a[FEATS].reset_index(drop=True).iloc[15:90720]); AEC[s] = np.load(OUT/'scores'/f'ae_ctx_{s}_N1.npz'); BUN[s] = c.load_bundle_from_training(OUT, s, DF, AEC[s]['scale'], N0[s], 'cpu')
    f = pd.read_parquet(L3/f'l3v3_{s}_CAL.parquet'); FR[s] = f[f.replica == 0].reset_index(drop=True).iloc[:AOLEN][FEATS]; e = pd.read_csv(L3/f'l3v3_{s}_CAL_events.csv'); EV[s] = e[e.replica == 0].reset_index(drop=True)
    for m in MODELS[1:]:
        z = np.load(OUT/'scores'/f'{FILES[m]}_{s}_{MODES[m]}.npz'); v = z['CAL_r0']; v = v[:, 0] if v.ndim == 2 else v; SAV[(m, s)] = c.nanw(v[:AOLEN])
        g = z['GEN_AO']; g = g[:, 0] if g.ndim == 2 else g; THR[(m, s)] = c.rolling_threshold(c.nanw(g[:AOLEN]), DF['models'][m]['q_T1_by_budget']['2'][s])
Z = {(m, s): np.asarray(B.normalise(FR[s], MODES[m], N0[s]), 'float64') for s in SYMBOLS for m in MODELS[1:]}
say('frozen models loaded; declared XGBoost ' + DF['models'][MODELS[1]]['configuration'] + ' (' + MODES[MODELS[1]] + '), Transformer ' + DF['models'][MODELS[2]]['configuration'] + ' (' + MODES[MODELS[2]] + '); CAL-3 events per symbol ' + str({s: len(EV[s]) for s in SYMBOLS}))

frozen models loaded; declared XGBoost A|pooled|d3 (N0), Transformer W16_d32_L1 (N1); CAL-3 events per symbol {'BTCUSDT': 200, 'ETHUSDT': 200}


## 2 · Explanation and deletion functions

In [ ]:
def xgb_x(s, z, b):
    """53 features of the last row of a short normalised slice ending at bar b"""
    return c.build_feats(z[b - 120:b + 1])[0][-1:]
def xgb_margin(s, X):
    dm = xgb.DMatrix(X, feature_names=c.FA_NAMES); return float(np.mean([b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b_, bi in BUN[s].xgb], axis=0)[0])
def xgb_shap(s, X):
    dm = xgb.DMatrix(X, feature_names=c.FA_NAMES); con = np.mean([b_.predict(dm, pred_contribs=True, iteration_range=(0, bi + 1)) for b_, bi in BUN[s].xgb], axis=0)[0]; return con[:-1], con[-1]
def shap6(con): return np.array([con[GIDX[g_]].sum() for g_ in FEATS])
def delete_xgb(s, z, b, j):
    zz = z[b - 120:b + 1].copy(); zz[-DELLEN:, j] = 0.0; return c.build_feats(zz)[0][-1:]
def tr_out(s, win):
    with torch.no_grad(): x = torch.tensor(np.asarray(win, 'float32')[None]); return float(np.mean([n_(x).float().item() for n_ in BUN[s].tr_nets]))
def tr_gradxinput(s, win):
    """mean over the declared nets of d output / d input, times input; returns (W, 6) and the mean gradient"""
    x = torch.tensor(np.asarray(win, 'float32')[None]); x.requires_grad_(True); y = 0.0
    for n_ in BUN[s].tr_nets: y = y + n_(x).sum() / len(BUN[s].tr_nets)
    y.backward(); g = x.grad[0].numpy().astype('float64'); return g * np.asarray(win, 'float64'), g
def delete_tr(win, j):
    w2 = np.asarray(win, 'float32').copy(); w2[-DELLEN:, j] = 0.0; return w2
def explain_tgnn(*args, **kwargs):
    """Interface for the TGNN explanation. Nothing is produced until the TGNN is integrated and tested."""
    return {'available': False, 'reason': 'TGNN is not integrated yet; no explanation is produced or claimed', 'method': None, 'feature_attribution': None, 'time_attribution': None, 'faithfulness_tested': False}

## 3 · Model-active events

In [ ]:
def peak_bar(s, m, ev):
    sc = SAV[(m, s)]; ii = np.arange(ev.t0, ev.t0 + ev.dur); v = sc[ii]; v = np.where(np.isfinite(v), v, -np.inf); return int(ii[int(np.argmax(v))]) if np.isfinite(v).any() and v.max() > -np.inf else None
XE = []; TE = []; excl = {'XGBoost v2': 0, 'Transformer v2': 0}; W_ = BUN[SYMBOLS[0]].tr_W
for s in SYMBOLS:
    for ev in EV[s].itertuples():
        for m, store in [(MODELS[1], XE), (MODELS[2], TE)]:
            b = peak_bar(s, m, ev)
            if b is None or b < PRE: excl[m] += 1; continue
            if not (SAV[(m, s)][b] > THR[(m, s)][b]): excl[m] += 1; continue
            store.append((s, ev.family, ev.amp, b))
say(f'model-active events: XGBoost {len(XE)} (excluded {excl[MODELS[1]]}), Transformer {len(TE)} (excluded {excl[MODELS[2]]}) of {sum(len(v) for v in EV.values())}; Transformer window {W_} bars')

model-active events: XGBoost 288 (excluded 112), Transformer 280 (excluded 120) of 400; Transformer window 16 bars


## 4 · Unit tests

In [ ]:
s0 = SYMBOLS[0]; s_, f_, a_, b_ = XE[0] if XE else (s0, 1, 0, 2000); zx = Z[(MODELS[1], s_)]; X = xgb_x(s_, zx, b_); con, bias = xgb_shap(s_, X)
assert abs(con.sum() + bias - xgb_margin(s_, X)) <= 1e-3; say('PASS T01 SHAP additivity: contributions + bias equal the booster margin (first event)')
errs = []
for (s_, f_, a_, b_) in XE[:20]: errs.append(abs(xgb_margin(s_, xgb_x(s_, Z[(MODELS[1], s_)], b_)) - SAV[(MODELS[1], s_)][b_]))
assert max(errs) <= 1e-3, max(errs); say(f'PASS T02 XGBoost margin recomputed from a short slice equals the saved score on {len(errs)} event peak bars (max difference {max(errs):.1e})')
errs = []
for (s_, f_, a_, b_) in TE[:20]: errs.append(abs(tr_out(s_, Z[(MODELS[2], s_)][b_ - W_ + 1:b_ + 1]) - SAV[(MODELS[2], s_)][b_]))
assert max(errs) <= 5e-3, max(errs); say(f'PASS T03 Transformer output recomputed here equals the saved score on {len(errs)} event peak bars (max difference {max(errs):.1e}; tolerance 5e-3 because the saved scores come from another device, protocol amendment 1)')
rg = np.random.default_rng(0); rel = []
for (s_, f_, a_, b_) in TE[:20]:
    win = Z[(MODELS[2], s_)][b_ - W_ + 1:b_ + 1]; ai, g = tr_gradxinput(s_, win); d = rg.normal(size=win.shape) * 1e-2; lin = float((g * d).sum()); act = tr_out(s_, win + d) - tr_out(s_, win)
    if abs(lin) > 1e-6: rel.append(abs(act - lin) / abs(lin))
assert len(rel) and np.median(rel) <= 0.05, np.median(rel); say(f'PASS T04 gradient first-order check: median relative error {np.median(rel):.3f} over {len(rel)} windows')
win = Z[(MODELS[2], s_)][b_ - W_ + 1:b_ + 1] if TE else np.zeros((W_, 6)); w2 = delete_tr(win, 2); assert np.allclose(np.delete(w2, 2, axis=1), np.delete(win.astype('float32'), 2, axis=1)) and (w2[-DELLEN:, 2] == 0).all(); say('PASS T05 deleting feature j changes only column j (and only the last 64 bars)')
tg = explain_tgnn(); assert tg['available'] is False and tg['feature_attribution'] is None and tg['time_attribution'] is None; say('PASS T06 the TGNN explanation interface returns available = false and no numbers')

PASS T01 SHAP additivity: contributions + bias equal the booster margin (first event)
PASS T02 XGBoost margin recomputed from a short slice equals the saved score on 20 event peak bars (max difference 0.0e+00)
PASS T03 Transformer output recomputed here equals the saved score on 20 event peak bars (max difference 1.7e-03; tolerance 5e-3 because the saved scores come from another device, protocol amendment 1)
PASS T04 gradient first-order check: median relative error 0.019 over 20 windows
PASS T05 deleting feature j changes only column j (and only the last 64 bars)
PASS T06 the TGNN explanation interface returns available = false and no numbers


## 5 · Deletion faithfulness

In [ ]:
def spear(a, b):
    if np.std(a) == 0 or np.std(b) == 0: return 0.0
    return float(spearmanr(a, b)[0])
rows = []; rg = np.random.default_rng(7)
for (s, fam, amp, b) in XE:
    z = Z[(MODELS[1], s)]; X = xgb_x(s, z, b); con, bias = xgb_shap(s, X); sh = shap6(con); m0 = xgb_margin(s, X); D = np.array([m0 - xgb_margin(s, delete_xgb(s, z, b, j)) for j in range(6)])
    rows.append({'model': 'XGBoost v2', 'sym': s, 'family': fam, 'attr': sh, 'D': D, 'rand': rg.permutation(6).astype(float)})
Wd = BUN[SYMBOLS[0]].tr_W
for (s, fam, amp, b) in TE:
    z = Z[(MODELS[2], s)]; win = z[b - Wd + 1:b + 1]; ai, g = tr_gradxinput(s, win); at = np.abs(ai).sum(axis=0); o0 = tr_out(s, win); D = np.array([o0 - tr_out(s, delete_tr(win, j)) for j in range(6)])
    rows.append({'model': 'Transformer v2', 'sym': s, 'family': fam, 'attr': at, 'D': D, 'rand': rg.permutation(6).astype(float), 'last10': float(np.abs(ai)[-10:].sum() / max(np.abs(ai).sum(), 1e-12))})
R = pd.DataFrame(rows)
def boot_mean(x, n=NBOOT, seed=0):
    x = np.asarray(x, float); rg_ = np.random.default_rng(seed); bs = [x[rg_.integers(0, len(x), len(x))].mean() for _ in range(n)]; return np.percentile(bs, [2.5, 97.5])
OUTR = {}; tab = []
for m in ['XGBoost v2', 'Transformer v2']:
    r = R[R.model == m]
    if len(r) == 0: OUTR[m] = None; continue
    hit = np.array([float(np.argmax(a) == np.argmax(d)) for a, d in zip(r.attr, r.D)]); rho = np.array([spear(a, d) for a, d in zip(r.attr, r.D)]); hitr = np.array([float(np.argmax(a) == np.argmax(d)) for a, d in zip(r.rand, r.D)]); rhor = np.array([spear(a, d) for a, d in zip(r.rand, r.D)])
    ci = boot_mean(rho); ok = bool(hit.mean() >= 0.50 and rho.mean() >= 0.40 and ci[0] > 0); OUTR[m] = dict(n=len(r), hit=float(hit.mean()), rho=float(rho.mean()), ci=[float(x) for x in ci], hold=ok, hit_rand=float(hitr.mean()), rho_rand=float(rhor.mean()))
    tab.append({'model': m, 'active events': len(r), 'top-1 = largest drop': hit.mean(), 'chance (1/6)': 1 / 6, 'random-attribution top-1': hitr.mean(), 'mean Spearman(attribution, drop)': rho.mean(), 'interval low': ci[0], 'interval high': ci[1], 'random-attribution Spearman': rhor.mean(), 'mean drop of top-1 feature': float(np.mean([d[np.argmax(a)] for a, d in zip(r.attr, r.D)])), 'mean drop over the 6 features': float(np.mean([d.mean() for d in r.D]))})
TB = pd.DataFrame(tab); TB.to_csv(OUT/'e2_faithfulness_table.csv', index=False); say(''); say('--- deletion faithfulness on model-active CAL-3 events ---'); say(TB.round(3).to_string(index=False))
FAMN = B.FAMS; frows = []
for m in ['XGBoost v2', 'Transformer v2']:
    r = R[R.model == m]
    for f in sorted(r.family.unique()):
        q = r[r.family == f]; sig = SIGN[int(f)]; top = [FEATS[int(np.argmax(a))] for a in q.attr]; frows.append({'model': m, 'family': FAMN[int(f)], 'events': len(q), 'top-1 = largest drop': float(np.mean([np.argmax(a) == np.argmax(d) for a, d in zip(q.attr, q.D)])), 'top-1 in expected signature': float(np.mean([t in sig for t in top])), 'chance (signature size / 6)': len(sig) / 6})
FF = pd.DataFrame(frows); FF.to_csv(OUT/'e2_family_table.csv', index=False); say(''); say('--- per family ---'); say(FF.round(2).to_string(index=False))
if OUTR.get('Transformer v2'): say(f'Transformer attribution mass in the last 10 bars of the window (descriptive): {float(R[R.model == "Transformer v2"].last10.mean()):.2f} (uniform would be {10 / Wd:.2f})')


--- deletion faithfulness on model-active CAL-3 events ---
         model  active events  top-1 = largest drop  chance (1/6)  random-attribution top-1  mean Spearman(attribution, drop)  interval low  interval high  random-attribution Spearman  mean drop of top-1 feature  mean drop over the 6 features
    XGBoost v2            288                 0.365         0.167                     0.174                             0.375         0.316           0.43                       -0.004                       1.487                          0.185
Transformer v2            280                 0.489         0.167                     0.161                             0.256         0.212           0.30                       -0.038                       2.529                          0.908

--- per family ---
         model                    family  events  top-1 = largest drop  top-1 in expected signature  chance (signature size / 6)
    XGBoost v2        pump_and_dump_like      34              

## 6 · Results (protocol E2) and save

In [ ]:
say(''); say('=== pre-registered hypotheses (protocol E2 v1b; CAL-3, development data) ===')
HC5 = bool(OUTR.get('XGBoost v2') and OUTR['XGBoost v2']['hold']); HC9 = bool(OUTR.get('Transformer v2') and OUTR['Transformer v2']['hold'])
for h, m, nm in [('H-C5', 'XGBoost v2', 'XGBoost TreeSHAP'), ('H-C9', 'Transformer v2', 'Transformer gradient x input')]:
    o = OUTR.get(m)
    if o is None: say(f'{h} {nm}: no model-active events -> FAILS'); continue
    say(f'{h} {nm} faithful (top-1 = largest drop >= 0.50 and mean Spearman >= 0.40 with lower bound > 0): top-1 hit {o["hit"]:.3f} (chance 0.167), Spearman {o["rho"]:.3f} ({o["ci"][0]:.3f}..{o["ci"][1]:.3f}) -> {"HOLDS" if o["hold"] else "FAILS"}')
sigT = FF[FF.model == 'Transformer v2']; hc10 = float(np.average(sigT['top-1 in expected signature'], weights=sigT['events'])) if len(sigT) else float('nan'); ch10 = float(np.average(sigT['chance (signature size / 6)'], weights=sigT['events'])) if len(sigT) else float('nan')
say(f'H-C10 (descriptive) Transformer top-1 in the expected signature: {hc10:.3f} versus chance {ch10:.3f}'); say('H-C11 exactness: SHAP additivity, saved-score parity (XGBoost, Transformer) and the gradient first-order check passed in T01-T04.')
HYP = {'H-C5': HC5, 'H-C9': HC9, 'H-C11': True}; say(f'{sum(HYP.values())} of 3 hold. Failed hypotheses are reported as failed; nothing was retuned. Explanations describe the model score, not the cause of a market event.')
DECL = {'status': 'DECLARED: explanation methods used by the engine and the dashboard', 'declared_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z', 'protocol_e2_v1b_sha256': PHE,
        'methods': {'xgboost': 'TreeSHAP of the declared boosters summed to six base features (nb45)', 'autoencoder': 'exact per-feature scaled errors (nb45)', 'transformer': 'gradient x input w.r.t. the normalised input window, summed |.| per feature (nb51)', 'tgnn': 'interface only: available = false'},
        'faithfulness_cal3': {m: OUTR[m] for m in OUTR if OUTR[m]}, 'hypotheses': HYP, 'deletion': {'replacement': 0.0, 'bars': DELLEN}, 'tgnn_interface': explain_tgnn(), 'mock': MOCK}
(OUT/'declared_explain_v2.json').write_text(json.dumps(DECL, indent=1, default=str)); (OUT/'e2_share_block.txt').write_text('\n'.join(LINES))
say('Saved: declared_explain_v2.json, e2_faithfulness_table.csv, e2_family_table.csv, e2_share_block.txt in models_b2_v1/ . Send the share block to Claude.')


=== pre-registered hypotheses (protocol E2 v1b; CAL-3, development data) ===
H-C5 XGBoost TreeSHAP faithful (top-1 = largest drop >= 0.50 and mean Spearman >= 0.40 with lower bound > 0): top-1 hit 0.365 (chance 0.167), Spearman 0.375 (0.316..0.430) -> FAILS
H-C9 Transformer gradient x input faithful (top-1 = largest drop >= 0.50 and mean Spearman >= 0.40 with lower bound > 0): top-1 hit 0.489 (chance 0.167), Spearman 0.256 (0.212..0.300) -> FAILS
H-C10 (descriptive) Transformer top-1 in the expected signature: 0.875 versus chance 0.617
H-C11 exactness: SHAP additivity, saved-score parity (XGBoost, Transformer) and the gradient first-order check passed in T01-T04.
1 of 3 hold. Failed hypotheses are reported as failed; nothing was retuned. Explanations describe the model score, not the cause of a market event.
Saved: declared_explain_v2.json, e2_faithfulness_table.csv, e2_family_table.csv, e2_share_block.txt in models_b2_v1/ . Send the share block to Claude.
